# Sprint 4 — Webinar 16 (Práctico) — Student Version

**Modalidad:** Coding Together + Breakout Rooms  
**Tema:** Funnels, CTEs y escenarios de impacto (SQLite)

> **Propósito de esta versión:** este cuaderno está diseñado para que tomes apuntes, escribas tus consultas, registres resultados y construyas conclusiones de negocio durante la sesión.

---

## 0. Objetivos de aprendizaje

Marca al final de la sesión lo que lograste.

| Objetivo | ¿Lo logré? | Mis notas |
|---|---|---|
| Construir un funnel de conversión usando CTEs encadenadas | ☐ | |
| Calcular tasas de conversión por etapa y por variante A/B | ☐ | |
| Simular una mejora de conversión y estimar su impacto económico | ☐ | |
| Traducir resultados técnicos a recomendaciones para negocio | ☐ | |

---

## 0.1 Agenda de trabajo

| Minuto | Actividad | Modalidad | Qué debo producir |
|---|---|---|---|
| 0–10 | Contexto + setup | Plenaria | Entorno listo y dataset cargado |
| 10–30 | Funnel base por variante | Coding Together | Query + tasas de conversión |
| 30–35 | Puesta en común | Plenaria | Observaciones iniciales |
| 35–55 | Simulación de uplift | Coding Together + Breakout Rooms | Escenario cuantificado |
| 55–75 | Segmentos + tablero ejecutivo | Breakout Rooms + Plenaria | Tabla final + recomendación |
| 75–80 | Cierre | Plenaria | Reflexión y siguientes pasos |

---

## 0.2 Cómo usar este notebook

En esta sesión trabajaremos con **SQLite online**. Usa este cuaderno como guía de trabajo:

1. Ejecuta primero el bloque **DDL**.
2. Antes de escribir SQL, completa las tablas de predicción o notas.
3. Resuelve cada ejercicio en las **celdas vacías**.
4. Copia tus resultados más importantes en las secciones de interpretación.
5. Cierra cada bloque con una conclusión de negocio, no solo técnica.

### Checklist inicial
- ☐ Abrí mi motor SQLite online.
- ☐ Verifiqué que el motor seleccionado sea **SQLite**.
- ☐ Ejecuté el script DDL.
- ☐ Puedo consultar la tabla `funnel_sessions`.
- ☐ Tengo claro qué mide cada columna del funnel.

---

## 1. Contexto del caso

Estás analizando un experimento A/B sobre un funnel de e-commerce.

### Journey analizado
`viewed_product → added_to_cart → begin_checkout → purchased`

### Tablas disponibles
- `users`: contiene atributos del usuario.
- `funnel_sessions`: contiene sesiones, variante, pasos del funnel y revenue.

### Preguntas del negocio
1. ¿Qué variante convierte mejor?
2. ¿En qué etapa está el principal cuello de botella?
3. ¿Qué pasaría si mejoramos una etapa específica?
4. ¿El impacto es igual para todos los segmentos?

Completa antes de empezar:

| Pregunta | Mi hipótesis inicial |
|---|---|
| ¿La variante A o B tendrá mejor conversión final? | |
| ¿Qué etapa parece más crítica? | |
| ¿Qué segmento podría responder mejor a una mejora? | |

---

## 1.1 Recordatorio rápido

### Funnel
Un funnel representa una secuencia de pasos que lleva a una conversión final.

### CTE
Una **CTE** permite dividir una consulta en bloques intermedios reutilizables y más legibles.

### Uplift
En esta sesión trabajaremos con una mejora expresada en **puntos porcentuales**.

Completa con tus palabras:

| Concepto | Definición / explicación personal |
|---|---|
| Funnel | |
| Tasa de conversión | |
| CTE | |
| Uplift de +5 p.p. | |

---

## 1.2 Script DDL de la sesión (ejecutar primero)

Ejecuta este bloque completo en SQLite antes de resolver cualquier ejercicio.

> **Objetivo:** crear las tablas `users` y `funnel_sessions` con un dataset pequeño para practicar análisis de funnel, escenarios y segmentación.


In [ ]:
-- ==============================================
-- DDL S4W14 para SQLiteOnline (sin esquemas)
-- Pega y ejecuta este bloque primero.
-- ==============================================

PRAGMA foreign_keys = ON;

DROP TABLE IF EXISTS funnel_sessions;
DROP TABLE IF EXISTS users;

CREATE TABLE users (
  user_id INTEGER PRIMARY KEY,
  segment TEXT NOT NULL,
  country TEXT NOT NULL
);

CREATE TABLE funnel_sessions (
  session_id INTEGER PRIMARY KEY,
  user_id INTEGER NOT NULL,
  variant TEXT NOT NULL,
  viewed_product INTEGER NOT NULL,
  added_to_cart INTEGER NOT NULL,
  begin_checkout INTEGER NOT NULL,
  purchased INTEGER NOT NULL,
  revenue REAL NOT NULL,
  session_date DATE NOT NULL,
  FOREIGN KEY (user_id) REFERENCES users(user_id)
);

INSERT INTO users (user_id, segment, country) VALUES
  (1, 'web', 'CO'),
  (2, 'mobile', 'CO'),
  (3, 'web', 'CO'),
  (4, 'web', 'AR'),
  (5, 'web', 'AR'),
  (6, 'mobile', 'CO'),
  (7, 'web', 'CO'),
  (8, 'web', 'CO'),
  (9, 'web', 'AR'),
  (10, 'web', 'AR'),
  (11, 'mobile', 'CO'),
  (12, 'mobile', 'AR');

INSERT INTO funnel_sessions (
  session_id, user_id, variant,
  viewed_product, added_to_cart, begin_checkout, purchased,
  revenue, session_date
) VALUES
  (1, 1, 'A', 1,1,1,1, 75.0, '2025-10-05'),
  (2, 1, 'A', 1,0,0,0, 0.0, '2025-10-11'),
  (3, 2, 'A', 1,1,1,1, 75.0, '2025-10-02'),
  (4, 3, 'A', 1,0,0,0, 0.0, '2025-10-13'),
  (5, 3, 'A', 1,1,0,0, 0.0, '2025-10-20'),
  (6, 4, 'A', 1,1,1,1, 75.0, '2025-10-03'),
  (7, 4, 'A', 1,0,0,0, 0.0, '2025-10-13'),
  (8, 5, 'B', 1,1,1,1, 60.0, '2025-10-09'),
  (9, 5, 'A', 1,1,1,0, 0.0, '2025-10-06'),
  (10, 6, 'B', 1,1,0,0, 0.0, '2025-10-18'),
  (11, 6, 'A', 1,1,0,0, 0.0, '2025-10-02'),
  (12, 7, 'A', 1,0,0,0, 0.0, '2025-10-13'),
  (13, 8, 'A', 1,1,0,0, 0.0, '2025-10-11'),
  (14, 8, 'A', 1,1,1,0, 0.0, '2025-10-15'),
  (15, 9, 'B', 1,1,0,0, 0.0, '2025-10-18'),
  (16, 10, 'B', 1,0,0,0, 0.0, '2025-10-13'),
  (17, 10, 'B', 1,1,0,0, 0.0, '2025-10-17'),
  (18, 11, 'A', 1,1,0,0, 0.0, '2025-10-05'),
  (19, 11, 'A', 1,1,1,1, 90.0, '2025-10-20'),
  (20, 12, 'B', 1,0,0,0, 0.0, '2025-10-01'),
  (21, 12, 'A', 1,1,1,1, 75.0, '2025-10-04');

## 1.3 Sanity checks

Antes de avanzar, escribe tus predicciones y luego valida con SQL.

| Validación | Mi predicción |
|---|---|
| Total de usuarios en `users` | |
| Total de sesiones en `funnel_sessions` | |
| Variantes presentes | |
| Segmentos presentes | |

Después de ejecutar la validación, registra tus hallazgos:

| Hallazgo observado | ¿Era lo que esperabas? |
|---|---|
| | |
| | |



In [ ]:
-- Sanity checks
SELECT COUNT(*) AS total_users FROM users;

SELECT COUNT(*) AS total_sessions FROM funnel_sessions;

SELECT DISTINCT variant
FROM funnel_sessions
ORDER BY variant;

SELECT DISTINCT segment
FROM users
ORDER BY segment;

---
# 2. Ejercicio 1 — Funnel base y tasas de conversión por variante

## 2.1 Enunciado

Usando `funnel_sessions`, calcula por cada `variant`:

- sesiones que inician el funnel (`n_view`),
- sesiones que llegan a `added_to_cart`,
- sesiones que llegan a `begin_checkout`,
- sesiones que llegan a `purchased`,
- tasas:
  - `view → cart`
  - `cart → checkout`
  - `checkout → purchase`
  - `view → purchase`

## Antes de escribir SQL
Completa esta planeación:

| Paso | ¿Qué voy a calcular? | Columna(s) involucradas |
|---|---|---|
| Base del funnel | | |
| Agregación por variante | | |
| Tasas de conversión | | |

### Entregable esperado
Una tabla con una fila por variante y las métricas del funnel completo.


In [ ]:
-- Ejercicio 1
-- Escribe aquí tu solución.
-- Recomendación: usa al menos 2 CTEs.
-- Puedes partir de esta idea:
--
-- WITH base AS (
--   SELECT ...
-- ),
-- agg AS (
--   SELECT ...
-- )
-- SELECT ...
;

## 2.2 Registro de resultados

Completa con tus resultados más importantes.

| variant | n_view | n_cart | n_checkout | n_purchase | view_cart | cart_checkout | checkout_purchase | view_purchase |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| A | | | | | | | | |
| B | | | | | | | | |

### Interpretación
1. ¿Qué variante tiene mejor conversión final?  
**Respuesta:** 

2. ¿En qué etapa observas la mayor caída?  
**Respuesta:** 

3. ¿Qué le dirías a un PM en una sola frase?  
**Respuesta:** 


## 2.3 Pistas opcionales

Usa estas pistas solo si las necesitas.

- Filtra a las sesiones que realmente **inician** el funnel (`viewed_product = 1`).
- Para contar sesiones que avanzan a una etapa, puedes usar:
  - `SUM(CASE WHEN ... THEN 1 ELSE 0 END)`
- Para las tasas:
  - numerador = etapa siguiente
  - denominador = etapa anterior
- Recuerda evitar división entera en SQLite usando `1.0 * ...`

---

# 3. Ejercicio 2 — Simular una mejora en una etapa del funnel

## 3.1 Enunciado

Toma la **variante B** como la experiencia nueva y simula una mejora de **+5 puntos porcentuales** en la tasa `checkout → purchase`.

### Supuestos
- `n_checkout` se mantiene constante.
- El ticket promedio por compra se mantiene constante.
- Solo cambia la tasa `checkout → purchase`.

## Antes de escribir SQL
Completa la lógica esperada:

| Pregunta | Mi respuesta |
|---|---|
| ¿Qué métricas necesito traer del funnel actual? | |
| ¿Cómo obtengo el ticket promedio? | |
| ¿Cómo convierto +5 p.p. en una nueva tasa? | |
| ¿Cómo estimaré compras incrementales? | |

### Fórmulas guía
- `new_checkout_purchase_rate = current_rate + 0.05`
- `projected_purchases = n_checkout * new_rate`
- `incremental_purchases = projected_purchases - current_purchases`
- `incremental_revenue = incremental_purchases * avg_revenue_per_purchase`


In [ ]:
-- Ejercicio 2
-- Escribe aquí tu solución.
-- Sugerencia:
-- 1) Construye métricas base por variante
-- 2) Quédate con variant = 'B'
-- 3) Calcula current_rate, new_rate, projected_purchases e incremental_revenue

;

## 3.2 Registro del escenario

Completa con tus resultados.

| Métrica | Valor |
|---|---:|
| Variante analizada | B |
| `n_checkout` actual | |
| `n_purchase` actual | |
| `checkout_purchase_rate` actual | |
| `avg_revenue_per_purchase` | |
| `new_checkout_purchase_rate` | |
| `projected_purchases` | |
| `incremental_purchases` | |
| `incremental_revenue` | |

### Interpretación de negocio
1. ¿La mejora produce un impacto material o marginal?  
**Respuesta:** 

2. ¿Qué supuestos vuelven frágil esta simulación?  
**Respuesta:** 

3. ¿Qué dato adicional pedirías antes de aprobar una decisión?  
**Respuesta:** 


## 3.3 Validación conceptual

Marca lo que verificaste al terminar tu query:

- ☐ La nueva tasa no supera 1.00.
- ☐ El ticket promedio se calculó usando solo compras reales.
- ☐ La simulación usa la variante correcta.
- ☐ Distingo entre **puntos porcentuales** y **porcentaje relativo**.
- ☐ Mi conclusión no ignora los supuestos del escenario.

---

# 4. Ejercicio 3 — Segmentos, tablero ejecutivo y recomendaciones

## 4.1 Enunciado

Ahora analiza la performance por combinación de `segment` y `variant`.

Calcula:

- `n_view`
- `n_purchase`
- `conv_view_purchase`
- `revenue_total`

### Tabla objetivo
| segment | variant | n_view | n_purchase | conv_view_purchase | revenue_total |

## Planeación
Antes de escribir SQL, responde:

| Decisión | Mi respuesta |
|---|---|
| ¿Qué tabla aporta el segmento? | |
| ¿Cuál es la llave del JOIN? | |
| ¿Qué filtro debo mantener para definir el inicio del funnel? | |
| ¿Qué agregaciones necesito? | |


In [ ]:
-- Ejercicio 3
-- Escribe aquí tu solución.
-- Sugerencia:
-- 1) JOIN entre users y funnel_sessions
-- 2) filtro de sesiones que inician el funnel
-- 3) agregación por segment y variant
-- 4) cálculo de conversión final

;

## 4.2 Tablero ejecutivo

Copia aquí los resultados más importantes de tu tabla final.

| segment | variant | n_view | n_purchase | conv_view_purchase | revenue_total |
|---|---|---:|---:|---:|---:|
| web | A | | | | |
| web | B | | | | |
| mobile | A | | | | |
| mobile | B | | | | |

### Lectura ejecutiva
1. ¿Qué combinación `segment + variant` muestra mejor desempeño?  
**Respuesta:** 

2. ¿Dónde conviene priorizar una mejora primero?  
**Respuesta:** 

3. ¿Qué combinación parece menos eficiente?  
**Respuesta:** 


## 4.3 Recomendación para stakeholders

Escribe una recomendación ejecutiva breve.

### Plantilla sugerida
> Con base en el análisis del funnel, recomendaría priorizar __________ en el segmento __________ porque __________.  
> El impacto esperado sería __________, aunque debemos validar __________ antes de escalar la decisión.

### Mi versión final


In [ ]:
# Opcional: usa esta celda como borrador de texto
recomendacion = 
print(recomendacion)

---
# 5. Retos opcionales

Resuelve estos retos solo si terminas antes.

## Reto 1 — Cambiar la etapa del uplift
Simula ahora una mejora de **+7 p.p.** en `view → cart` para la variante `A`.

### Antes de empezar
- ¿Qué métricas cambiarían?
- ¿Qué supuesto adicional estás haciendo?


In [ ]:
-- Reto 1
-- Simulación alternativa: uplift en view -> cart para A

;

## Reto 2 — Priorización por país

Extiende el tablero ejecutivo para incluir `country` además de `segment` y `variant`.

### Preguntas guía
1. ¿Aparecen diferencias relevantes por país?
2. ¿La misma variante funciona igual en todos los contextos?
3. ¿Qué combinación sería tu prioridad de negocio?


In [ ]:
-- Reto 2
-- Tablero por country + segment + variant

;

## 6. Errores comunes de esta sesión

Marca los que te pasaron y anota cómo los resolviste.

| Error común | ¿Me pasó? | ¿Cómo lo resolví? |
|---|---|---|
| Olvidé ejecutar el DDL | ☐ | |
| Hice el JOIN con una llave incorrecta | ☐ | |
| Calculé mal el denominador de una tasa | ☐ | |
| Confundí p.p. con % relativo | ☐ | |
| Redacté una conclusión técnica pero no de negocio | ☐ | |

---

## 7. Cierre y autoevaluación

### Lo más importante que me llevo hoy
- 
- 
- 

### Lo que todavía necesito practicar
- 
- 
- 

### Reflexión final
1. ¿Qué recomendación le darías al Product Manager con base en este análisis?  
**Respuesta:** 

2. ¿Cómo comunicarías un resultado negativo o no concluyente de forma profesional?  
**Respuesta:** 

3. ¿Qué parte del ejercicio te resultó más difícil: SQL, interpretación o traducción a negocio?  
**Respuesta:** 


## 8. Recursos y siguientes pasos

### ¿Necesitas ayuda?
- `DATA-CO-LEARNING`: revisa los horarios de apoyo en la guía del estudiante.
- `DA_CONSULTA`: publica dudas de contenido o del proyecto usando el tag correspondiente.
- `SPRINT FOCUS`: sesiones extra para profundizar temas clave del sprint.
- `SESIONES 1:1`: agenda con anticipación si necesitas apoyo individual.

### Próxima sesión
**Sprint 5 — Introducción a Python y Pandas**

### Checklist de cierre
- ☐ Guardé mis consultas finales.
- ☐ Registré mis conclusiones.
- ☐ Sé explicar qué significa un uplift en puntos porcentuales.
- ☐ Estoy listo/a para pasar del análisis en SQL a análisis en Python.

---